In [1]:

%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

matplotlib.use('Qt5Agg')  # Asegúrate de que este backend está instalado.
mne.viz.set_browser_backend('qt')  # Activa el backend interactivo

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

from neurodsp.spectral import compute_spectrum, trim_spectrum
from neurodsp.plts import plot_power_spectra

# Import IRASA related functions
from neurodsp.aperiodic import compute_irasa, fit_irasa

from joblib import Parallel, delayed

Using qt as 2D backend.


In [2]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_pat

In [4]:
f_min=0.5
f_max=40


#epochs
combinaciones = ["zinnen", "woorden"]

subj = []
# Recorremos cada subdirectorio en la carpeta base
for subdirectorio in general_datadir.iterdir():
    # Comprobamos que el elemento sea un directorio y que su nombre comience con 'sub-A2'
    if modality == "visual":
        subject_prefix = 'sub-V1'
    elif modality == "auditory":
        subject_prefix = 'sub-A2'
        
    if subdirectorio.is_dir() and subdirectorio.name.startswith(subject_prefix):
        # Añadimos el nombre del sujeto a la lista
        subj.append(subdirectorio.name)

print(subj)
##tablas de canales

channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
channels_mag=channels[channels[f"canal_efectivo"].notna()][f"canal_efectivo"]
channels_mag=channels_mag.tolist()
del channels

['sub-V1001', 'sub-V1002', 'sub-V1003', 'sub-V1004', 'sub-V1005', 'sub-V1006', 'sub-V1007', 'sub-V1008', 'sub-V1009', 'sub-V1010', 'sub-V1011', 'sub-V1012', 'sub-V1013', 'sub-V1015', 'sub-V1016', 'sub-V1017', 'sub-V1019', 'sub-V1020', 'sub-V1022', 'sub-V1024', 'sub-V1025', 'sub-V1026', 'sub-V1027', 'sub-V1028', 'sub-V1029', 'sub-V1030', 'sub-V1031', 'sub-V1032', 'sub-V1033', 'sub-V1034', 'sub-V1035', 'sub-V1036', 'sub-V1037', 'sub-V1038', 'sub-V1039', 'sub-V1040', 'sub-V1042', 'sub-V1044', 'sub-V1045', 'sub-V1046', 'sub-V1048', 'sub-V1049', 'sub-V1050', 'sub-V1052', 'sub-V1053', 'sub-V1054', 'sub-V1055', 'sub-V1057', 'sub-V1058', 'sub-V1059', 'sub-V1061', 'sub-V1062', 'sub-V1063', 'sub-V1064', 'sub-V1065', 'sub-V1066', 'sub-V1068', 'sub-V1069', 'sub-V1070', 'sub-V1071', 'sub-V1072', 'sub-V1073', 'sub-V1074', 'sub-V1075', 'sub-V1076', 'sub-V1077', 'sub-V1078', 'sub-V1079', 'sub-V1080', 'sub-V1081', 'sub-V1083', 'sub-V1084', 'sub-V1085', 'sub-V1086', 'sub-V1087', 'sub-V1088', 'sub-V1089'

In [4]:
## obtener power spechtum de el objeto evoked -- esto serñía el MIXED POWER 

# psds=evoked.compute_psd(method='welch', fmin=fmin, fmax=fmax, reject_by_annotation=True, n_fft = int(6 * sfreq))
# psds_all.append(psds)
            
# psds,freqs= mne.time_frequency.psd_array_welch(array, sfreq, fmin=0, fmax=inf, n_fft=256, n_overlap=0, 
# #                                    n_per_seg=None, n_jobs=None, average='mean', window='hamming', remove_dc=True, *, 
# #                                    output='power', verbose=None)



##please note THAT THIS IS ONLY FOR EVOKED, YOU NEED TO CALCULATE IT THROUGH EPOCHS

# ##def compute_psd(self, fmin=0, fmax=np.inf, tmin=None, tmax=None, proj=False)
# array=evoked.get_data()
# sfreq=evoked.info['sfreq']
# fmin=0.5
# fmax=40
# nfft=1024 # to increase the spectral resolution, , although you can go to 4096 if you want to see more details
# njobs=10

# ##compute psd FOR WHOLE SIGNAL
# psds,freqs= mne.time_frequency.psd_array_welch(array, sfreq, fmin=fmin, fmax=fmax, n_fft=nfft, n_jobs=njobs)



In [5]:
def compute_ple(subj, epochs, condition, f_range=(0.5, 40), hset=None, thresh=None, isplot=False):
    # Obtener solo MEG sin canales marcados como bads
    data_epochs = epochs.copy().pick(picks="meg", exclude="bads").get_data()
    channels_mag = epochs.copy().pick(picks="meg", exclude="bads").ch_names

    sfreq = epochs.info['sfreq']
    duration = epochs.tmax - epochs.tmin

    # Contenedores globales
    psd_periodic_elect_all_epoch_all = []
    psd_aperiodic_elect_all_epoch_all = []
    slope_elect_all_epoch_all = []
    intercept_y_elect_all_epoch_all = []

    # Función paralelizable: IRASA + ajuste de pendiente
    def compute_irasa_fit(data_elect):
        freqs, psd_aperiodic, psd_periodic = compute_irasa(
            data_elect, fs=sfreq, f_range=f_range, hset=hset, thresh=thresh
        )
        intercept, slope = fit_irasa(freqs, psd_aperiodic)
        return psd_aperiodic, psd_periodic, intercept, slope, freqs

    # Procesar cada época
    for i, data_epoch in enumerate(data_epochs):
        # Paralelizar por canal
        results = Parallel(n_jobs=-1)(
            delayed(compute_irasa_fit)(data_epoch[ch])
            for ch in range(len(data_epoch))
        )

        # Extraer resultados
        psd_aperiodic_elect_all_epoch = [r[0] for r in results]
        psd_periodic_elect_all_epoch = [r[1] for r in results]
        intercept_y_elect_all_epoch = [r[2] for r in results]
        slope_elect_all_epoch = [r[3] for r in results]
        freqs = results[0][4]

        # Agregar por época
        psd_aperiodic_elect_all_epoch_all.append(psd_aperiodic_elect_all_epoch)
        psd_periodic_elect_all_epoch_all.append(psd_periodic_elect_all_epoch)
        intercept_y_elect_all_epoch_all.append(intercept_y_elect_all_epoch)
        slope_elect_all_epoch_all.append(slope_elect_all_epoch)

    # Construir DataFrame
    num_epochs = len(data_epochs)
    num_elects = len(channels_mag)
    shape_tabla = num_epochs * num_elects

    psd_periodic_elect_all_epoch_all_list = np.array(psd_periodic_elect_all_epoch_all).reshape(shape_tabla, -1).tolist()
    psd_aperiodic_elect_all_epoch_all_list = np.array(psd_aperiodic_elect_all_epoch_all).reshape(shape_tabla, -1).tolist()

    table_PLE = pd.DataFrame({
        'Subject': [subj] * shape_tabla,
        'Condition': [condition] * shape_tabla,
        'freqs': [freqs] * shape_tabla,
        'Epoch': np.repeat(np.arange(num_epochs), num_elects),
        'Elect': np.tile(channels_mag, num_epochs),
        'psd_periodic_elect_all_epoch_all': psd_periodic_elect_all_epoch_all_list,
        'psd_aperiodic_elect_all_epoch_all': psd_aperiodic_elect_all_epoch_all_list,
        'intercept_y_elect_all_epoch_all': np.array(intercept_y_elect_all_epoch_all).flatten(),
        'slope_elect_all_epoch_all': np.array(slope_elect_all_epoch_all).flatten()
    })

    return table_PLE



In [6]:
# ## compute power law exponent for each epoch
# def ple_exponent(psds_all, isplot=False):
#     ##prueba con epochs, ahora has de adaptarlo a la función de power_spectrum_nans 
#     # first you need to obtain the frequencies, 
#     freqs= psds_all[0].freqs
#     #for np.polyfit we need to convert the frequencies to array
#     psds_array = [psds.get_data() for psds in psds_all]
#     psds_avg = [np.mean(epoch_psd, axis=0) for epoch_psd in psds_array]
#     psds_grand_avg = np.mean(psds_avg, axis=0)

#     coeffs= np.polyfit(np.log10(freqs), np.log10(psds_grand_avg), deg=1)
#     PLE= -coeffs[0]
#     if isplot==True:
#         #plot of the power spectrum
#         plt.loglog(freqs, psds_grand_avg,  label='PSD Mean')
#         #plot of the coeffs calculated
#         #its 10**y, where y is the linear regression, ax+b, a is slope, x is log10(freqs) and b is the intercept
#         y=coeffs[1] + coeffs[0]*np.log10(freqs)
#         plt.loglog(freqs, 10**y, 'r--', label=f'Fit PLE = {PLE:.2f}')
#         plt.xlabel('Frequency (Hz)')
#         plt.ylabel('Power Spectral Density (dB/Hz)')
#         plt.title('Power Law Exponent Fit using MNE-Python')
#         plt.legend()
#         plt.grid(True)
#         plt.show()

#         plt.show()



#     return coeffs, PLE, psds_grand_avg

In [6]:
##codigo para agrupar todas las tablas

all_tables = []

for i in range(0,len(subj)):
    
    for h in range(0,len(combinaciones)):
        try:
            subject=subj[i]
            combinacion= combinaciones[h]
            path_epochs= epochs_clean_path / f"{subject}_epochs_{combinacion}_{layer_script}-epo.fif"
            epochs = mne.read_epochs(path_epochs)
            table_PLE=compute_ple(subject,epochs, condition=combinacion, f_range=(f_min, f_max),isplot=False)
            all_tables.append(table_PLE)
            del epochs
        except:
            print(f"Error en {subject} en {combinacion}")
            continue

table_PLE_subjects_all = pd.concat(all_tables, ignore_index=True)

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1001_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_16856\3001391442.py:3: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1001_epochs_woorden_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
20 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_16856\3001391442.py:3: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1002_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
22 matching events found
No baseline correction applied
0 projection items activated


...

C:\Users\UCM\AppData\Local\Temp\ipykernel_16856\3001391442.py:3: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1117_epochs_zinnen_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
24 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_16856\3001391442.py:3: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="meg", exclude="bads").get_data()


Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1117_epochs_woorden_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
19 matching events found
No baseline correction applied
0 projection items activated


C:\Users\UCM\AppData\Local\Temp\ipykernel_16856\3001391442.py:3: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  data_epochs = epochs.copy().pick(picks="meg", exclude="bads").get_data()


In [8]:
table_PLE_subjects_all

,Subject,Condition,freqs,Epoch,Elect,psd_periodic_elect_all_epoch_all,psd_aperiodic_elect_all_epoch_all,intercept_y_elect_all_epoch_all,slope_elect_all_epoch_all
0,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC11-4304,"[2.680178974171939e-28, 1.1405398764560063e-27...","[3.209103275500903e-27, 2.5695296452843454e-27...",-26.506501,-1.176693
1,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC12-4304,"[2.292580777795326e-28, 1.5809032454905314e-27...","[4.711772708421555e-27, 4.023640108366676e-27,...",-26.373425,-1.216540
2,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC13-4304,"[1.0439785835775173e-27, 2.7026713247875535e-2...","[5.736748103642964e-27, 5.361811946626482e-27,...",-26.228568,-1.329808
3,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC14-4304,"[1.2187840606717464e-27, 2.925518346996426e-27...","[7.062962040379582e-27, 6.767769772386197e-27,...",-26.084587,-1.408159
4,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC15-4304,"[1.0067953221739206e-27, 2.3835486817925347e-2...","[8.428644379826931e-27, 8.182740617744654e-27,...",-26.018282,-1.385185
...,...,...,...,...,...,...,...,...,...
224674,sub-V1031,woorden,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",22,MZF03-4304,"[8.22848997585494e-28, 1.0367993457523965e-27,...","[2.3983064273962673e-27, 2.5732868995144352e-2...",-26.538984,-1.170870
224675,sub-V1031,woorden,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",22,MZO01-4304,"[2.553506999779442e-27, 8.440934467231635e-28,...","[1.4847486846839542e-26, 1.6068312105728053e-2...",-25.388422,-1.447890
224676,sub-V1031,woorden,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",22,MZO02-4304,"[-6.510233828582363e-28, -1.0264083767087018e-...","[5.8290523841606514e-27, 6.4182201032840255e-2...",-25.648302,-1.360120
224677,sub-V1031,woorden,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",22,MZO03-4304,"[3.0776808007016103e-28, 2.8839726938030736e-2...","[1.733778194392117e-27, 2.0529929980186006e-27...",-26.253003,-1.211886


In [9]:
PLE_path


WindowsPath('g:/MOUS_204/MOUS_visual/output_analysis/analysis_block/PLE_block')

In [7]:
table_PLE_subjects_all.to_pickle(PLE_path / f"table_PLE_subjects_all_{layer_script}.pickle")

In [8]:
table_PLE_slope_intercept_subjects_all= table_PLE_subjects_all[['Subject', 'Condition', 'freqs', 'Epoch', 'Elect', 'intercept_y_elect_all_epoch_all', 'slope_elect_all_epoch_all']]

In [9]:
table_PLE_slope_intercept_subjects_all.to_pickle(PLE_path / f"table_PLE_slope_intercept_subjects_all_{layer_script}.pickle")

In [20]:
table_PLE_slope_intercept_subjects_all

,Subject,Condition,freqs,Epoch,Elect,intercept_y_elect_all_epoch_all,slope_elect_all_epoch_all
0,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC11-4304,-26.506501,-1.176693
1,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC12-4304,-26.373425,-1.216540
2,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC13-4304,-26.228568,-1.329808
3,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC14-4304,-26.084587,-1.408159
4,sub-V1001,zinnen,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",0,MLC15-4304,-26.018282,-1.385185
...,...,...,...,...,...,...,...
224674,sub-V1031,woorden,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",22,MZF03-4304,-26.538984,-1.170870
224675,sub-V1031,woorden,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",22,MZO01-4304,-25.388422,-1.447890
224676,sub-V1031,woorden,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",22,MZO02-4304,-25.648302,-1.360120
224677,sub-V1031,woorden,"[0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2...",22,MZO03-4304,-26.253003,-1.211886
